# EmoExpress: RAG and Grounded LLM Response Generation

## Objective

This notebook builds a PDF-based Retrieval-Augmented Generation pipeline
and uses the retrieved knowledge to generate:

1. An empathetic response
2. Practical recommendations
3. An encouraging caption
4. A safe image-generation prompt
5. Source references with document titles and page numbers

Import Libraries

In [1]:
from pathlib import Path
import json
import os
import shutil
import time

import pandas as pd

from dotenv import load_dotenv
from openai import OpenAI

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

C:\Users\Anne\AppData\Local\Temp\ipykernel_24948\900368413.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Configure project paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

KNOWLEDGE_BASE_DIR = (
    PROJECT_ROOT
    / "knowledge_base"
)

VECTOR_STORE_DIR = (
    PROJECT_ROOT
    / "vector_store"
    / "chroma_db"
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
GENERATED_OUTPUT_DIR = (
    OUTPUT_DIR
    / "generated_responses"
)

for directory in [
    VECTOR_STORE_DIR.parent,
    GENERATED_OUTPUT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Project root:", PROJECT_ROOT)
print("Knowledge base:", KNOWLEDGE_BASE_DIR)
print("Vector store:", VECTOR_STORE_DIR)

Project root: d:\FullStack_Academic\Final Capstone\EmoExpress
Knowledge base: d:\FullStack_Academic\Final Capstone\EmoExpress\knowledge_base
Vector store: d:\FullStack_Academic\Final Capstone\EmoExpress\vector_store\chroma_db


Load the OpenAI API key

In [3]:
load_dotenv(
    PROJECT_ROOT / ".env"
)

OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY was not found in the .env file."
    )

print("OpenAI API key loaded successfully.")

client = OpenAI(api_key=OPENAI_API_KEY)

OpenAI API key loaded successfully.


**PART A: RAG Knowledge Base**

**Inspect PDF files**

Find all PDFs

In [4]:
pdf_files = sorted(KNOWLEDGE_BASE_DIR.rglob("*.pdf"))

print("Number of PDF files:",len(pdf_files))

for pdf_path in pdf_files:
    print("-",pdf_path.relative_to(KNOWLEDGE_BASE_DIR),)

Number of PDF files: 80
- career_and_work\burnout_wellbeing.pdf
- career_and_work\manage_stress.pdf
- career_and_work\signs_strategies_burnout.pdf
- career_and_work\stress_finding_job.pdf
- career_and_work\stress_resilience.pdf
- career_and_work\tips_job_search.pdf
- career_and_work\work_life_balance.pdf
- career_and_work\workplace_stress_overview.pdf
- daily_life\digital_wellbeing.pdf
- daily_life\manage_screentime.pdf
- daily_life\sunshine_screentime.pdf
- daily_life\support_digital_wellness.pdf
- daily_life\time_management.pdf
- daily_life\wellness_online.pdf
- education\nature_based_mind_body.pdf
- education\reduce_anxiety.pdf
- education\skill_overcome_anxiety.pdf
- education\test_anxiety_effects.pdf
- education\test_anxiety_medical_students.pdf
- education\tip_for_exam.pdf
- family\family_based_interventions.pdf
- family\parents_caregivers.pdf
- family\parents_youth_mental_health.pdf
- family\positive_parents.pdf
- family\resources.pdf
- family\talking_with_children.pdf
- financi

Create a PDF inventory

In [5]:
pdf_inventory = []

for pdf_path in pdf_files:
    topic = pdf_path.parent.name

    pdf_inventory.append(
        {
            "file_name": pdf_path.name,
            "topic": topic,
            "relative_path": str(
                pdf_path.relative_to(
                    KNOWLEDGE_BASE_DIR
                )
            ),
            "file_size_kb": round(
                pdf_path.stat().st_size
                / 1024,
                2,
            ),
        }
    )

pdf_inventory_df = pd.DataFrame(
    pdf_inventory
)

pdf_inventory_df

,file_name,topic,relative_path,file_size_kb
0,burnout_wellbeing.pdf,career_and_work,career_and_work\burnout_wellbeing.pdf,519.72
1,manage_stress.pdf,career_and_work,career_and_work\manage_stress.pdf,155.52
2,signs_strategies_burnout.pdf,career_and_work,career_and_work\signs_strategies_burnout.pdf,246.39
3,stress_finding_job.pdf,career_and_work,career_and_work\stress_finding_job.pdf,693.53
4,stress_resilience.pdf,career_and_work,career_and_work\stress_resilience.pdf,159.74
...,...,...,...,...
75,mindfulness_practice.pdf,stress_and_anxiety,stress_and_anxiety\mindfulness_practice.pdf,226.48
76,positive_thinking.pdf,stress_and_anxiety,stress_and_anxiety\positive_thinking.pdf,238.58
77,prevention_anxiety.pdf,stress_and_anxiety,stress_and_anxiety\prevention_anxiety.pdf,21414.08
78,reduce_stress.pdf,stress_and_anxiety,stress_and_anxiety\reduce_stress.pdf,206.32


**Load PDF pages**

Create a PDF-loading function

In [6]:
def load_pdf_documents(knowledge_base_directory):
    """
    Load every PDF page and attach topic and source metadata.
    """

    loaded_documents = []
    failed_files = []

    pdf_paths = sorted(knowledge_base_directory.rglob("*.pdf"))

    for pdf_path in pdf_paths:
        topic = pdf_path.parent.name

        try:
            loader = PyPDFLoader(str(pdf_path))

            pages = loader.load()

            for page in pages:
                original_page = (page.metadata.get("page",0,))                     

                page.metadata.update({
                        "source_file": (pdf_path.name),                      
                        "document_title": (pdf_path.stem.replace("_", " ").replace("-", " ").title()),
                        "topic": topic,
                        "page_number": (int(original_page)+ 1),
                        "relative_path": str(pdf_path.relative_to(knowledge_base_directory))})

                loaded_documents.append(page)

        except Exception as error:
            failed_files.append(
                {
                    "file": str(pdf_path),
                    "error": str(error),
                }
            )

    return loaded_documents, failed_files

Load the documents:

In [7]:
pdf_pages, failed_pdf_files = (load_pdf_documents(KNOWLEDGE_BASE_DIR))

print("Loaded PDF pages:",len(pdf_pages))

print("Failed PDF files:",len(failed_pdf_files))

Loaded PDF pages: 1384
Failed PDF files: 1


In [8]:
# failed_pdf_files

Remove empty pages

In [9]:
usable_pages = [
    page
    for page in pdf_pages
    if page.page_content
    and page.page_content.strip()
]

empty_page_count = (
    len(pdf_pages)
    - len(usable_pages)
)

print("Usable pages:", len(usable_pages))
print("Empty pages removed:", empty_page_count)

Usable pages: 1370
Empty pages removed: 14


**Split pages into chunks**

Configure the text splitter

In [10]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150

text_splitter = (
    RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=[ "\n\n", "\n", ". ", " ", "", ],))

Split the pages

In [11]:
document_chunks = (text_splitter.split_documents(usable_pages))

print( "Number of document chunks:",len(document_chunks))

Number of document chunks: 4829


Add chunk IDs:

In [12]:
for chunk_index, chunk in enumerate(document_chunks):
    chunk.metadata["chunk_id"] = (f"chunk_{chunk_index:06d}" )

Inspect a chunk

In [13]:
if document_chunks:
    sample_chunk = document_chunks[0]

    print("Chunk metadata:")
    print(sample_chunk.metadata)

    print("\nChunk text:")
    print(sample_chunk.page_content)

Chunk metadata:
{'producer': 'Acrobat Distiller 22.0 (Windows)', 'creator': 'Arbortext Advanced Print Publisher 9.1.520/W Unicode', 'creationdate': '2023-02-03T09:27:47+00:00', 'author': 'Shaun Prentice, Taryn Elliott, Diana Dorstyn, Jill Benson', 'figures': '2', 'keywords': '', 'moddate': '2023-02-10T18:29:38+08:00', 'subject': '', 'tables': '4', 'title': 'Burnout, wellbeing and how they relate: A qualitative study in general practice trainees', 'trapped': 'False', 'wps-articledoi': '10.1111/medu.14931', 'wps-journaldoi': '10.1111/(ISSN)1365-2923', 'wps-proclevel': '3', 'words': '10053', 'source': 'd:\\FullStack_Academic\\Final Capstone\\EmoExpress\\knowledge_base\\career_and_work\\burnout_wellbeing.pdf', 'total_pages': 13, 'page': 0, 'page_label': '243', 'source_file': 'burnout_wellbeing.pdf', 'document_title': 'Burnout Wellbeing', 'topic': 'career_and_work', 'page_number': 1, 'relative_path': 'career_and_work\\burnout_wellbeing.pdf', 'chunk_id': 'chunk_000000'}

Chunk text:
RESEARCH

Analyze chunk lengths

In [14]:
chunk_statistics_df = pd.DataFrame(
    {
        "character_count": [
            len(chunk.page_content)
            for chunk in document_chunks
        ],
        "topic": [
            chunk.metadata["topic"]
            for chunk in document_chunks
        ],
        "source_file": [
            chunk.metadata["source_file"]
            for chunk in document_chunks
        ],
    }
)

chunk_statistics_df[
    "character_count"
].describe()

count    4829.000000
mean      845.491613
std       229.960137
min         1.000000
25%       876.000000
50%       950.000000
75%       978.000000
max      1000.000000
Name: character_count, dtype: float64

In [15]:
# Display chunk count per topic:
chunk_statistics_df["topic"].value_counts()

topic
general_support       1245
career_and_work        994
loneliness             392
personal_growth        330
stress_and_anxiety     330
grief_and_loss         277
education              273
financial_concerns     241
self_confidence        238
relationships          192
family                 152
health_and_welness     110
daily_life              55
Name: count, dtype: int64

**Build the Chroma vector store**

Initialize embeddings

In [16]:
EMBEDDING_MODEL = ("text-embedding-3-small")

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL, api_key=OPENAI_API_KEY,)

print( "Embedding model:",  EMBEDDING_MODEL,)

Embedding model: text-embedding-3-small


Build the vector database

In [17]:
def create_or_load_vector_store(
    chunks,
    embedding_model,
    persist_directory,
    rebuild=False,
):
    """
    Create or load the persistent Chroma vector store.
    """

    persist_directory = Path(persist_directory)

    if rebuild and persist_directory.exists():
        shutil.rmtree( persist_directory        )

    if persist_directory.exists():
        print( "Loading existing vector store..."  )

        return Chroma(
            persist_directory=str(persist_directory ),
            embedding_function=(embedding_model ), )

    if not chunks:
        raise ValueError("No document chunks are available." )

    print( "Creating a new vector store..."    )

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=str(persist_directory))

    vector_store.persist()

    return vector_store

In [18]:
print("Starting vector-store creation...")
print("Input chunks:", len(document_chunks))
print("Persist directory:", VECTOR_STORE_DIR)
# Create the database for the first run:
build_start_time = time.perf_counter()

vector_store = create_or_load_vector_store(
    chunks=document_chunks,
    embedding_model=embeddings,
    persist_directory=VECTOR_STORE_DIR,
    rebuild=True,
)

build_time = (time.perf_counter()- build_start_time)

print(f"Vector store build time: "f"{build_time:.2f} seconds")
print("Stored chunks:", vector_store._collection.count())

Starting vector-store creation...
Input chunks: 4829
Persist directory: d:\FullStack_Academic\Final Capstone\EmoExpress\vector_store\chroma_db
Creating a new vector store...
Vector store build time: 33.62 seconds
Stored chunks: 4829


C:\Users\Anne\AppData\Local\Temp\ipykernel_24948\772686451.py:33: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vector_store.persist()


In [19]:
# # Create the database for the later run:
# vector_store = create_or_load_vector_store(
#     chunks=document_chunks,
#     embedding_model=embeddings,
#     persist_directory=VECTOR_STORE_DIR,
#     rebuild=False,
# )

Verify the collection

In [20]:
collection_count = (vector_store._collection.count())

print("Stored chunks:", collection_count)

Stored chunks: 4829


**PART B: Retrieval**

Build the retrieval query

In [21]:
def build_retrieval_query(
    user_story,
    emotions,
    primary_topic,
    secondary_topic=None,
):
    """
    Combine the user story, emotion predictions, and topics
    into one semantic retrieval query.
    """

    if emotions:
        emotion_text = ", ".join(emotions)
    else:
        emotion_text = "unspecified"

    secondary_text = (secondary_topic if secondary_topic else "none")

    return f"""
            User situation:{user_story}
            Detected emotions:{emotion_text}
            Primary topic:{primary_topic}
            Secondary topic:{secondary_text}
            Information needed:
                Safe, practical, non-diagnostic guidance, coping strategies,
                constructive next steps, and supportive resources relevant
                to this specific situation.""".strip()

Create a topic-aware retrieval function

In [22]:
def retrieve_documents(
    vector_store,
    query,
    primary_topic,
    k=5,
):
    """
    Search the primary topic first, then use broader
    fallback searches when necessary.
    """

    retrieval_attempts = []

    # Search 1: primary topic
    primary_results = (
        vector_store
        .similarity_search_with_relevance_scores(
            query=query,
            k=k,
            filter={
                "topic": primary_topic
            },
        )
    )

    retrieval_attempts.append(
        {
            "strategy": "primary_topic",
            "results": primary_results,
        }
    )

    if primary_results:
        return primary_results, (
            "primary_topic"
        )

    # Search 2: general support
    general_results = (
        vector_store
        .similarity_search_with_relevance_scores(
            query=query,
            k=k,
            filter={
                "topic": "general_support"
            },
        )
    )

    retrieval_attempts.append(
        {
            "strategy": "general_support",
            "results": general_results,
        }
    )

    if general_results:
        return general_results, (
            "general_support"
        )

    # Search 3: all documents
    broad_results = (
        vector_store
        .similarity_search_with_relevance_scores(
            query=query,
            k=k,
        )
    )

    return broad_results, "all_topics"

**Format RAG context**

Create a context formatter

In [23]:
def format_retrieved_context(retrieved_results,):

    context_sections = []
    sources = []

    for rank, (document,relevance_score,)in enumerate(retrieved_results,start=1):

        title = document.metadata.get("document_title","Unknown document",)
      
        page = document.metadata.get("page_number","Unknown")

        topic = document.metadata.get("topic","Unknown")

        context_sections.append(f"""          
                                SOURCE {rank}
                                Title: {title}
                                Page: {page}
                                Topic: {topic}
                                Relevance score: {float(relevance_score):.4f}
                                Passage:{document.page_content}""".strip())

        sources.append({
                "rank": rank,
                "title": title,
                "page": page,
                "topic": topic,
                "source_file": (document.metadata.get("source_file")),
                "relevance_score": float(relevance_score)})

    return ("\n\n".join(context_sections),sources)

**PART C: LLM Response**

System prompt

In [24]:
RESPONSE_SYSTEM_PROMPT = """
You are EmoExpress, an empathetic emotional-support assistant.

Generate a supportive response using the user's story, emotion
classification, topic classification, and retrieved knowledge.

Rules:

1. Do not diagnose medical or mental-health conditions.
2. Do not recommend medication changes.
3. Do not claim to be a therapist or physician.
4. Do not invent document titles, page numbers, or citations.
5. Use retrieved passages only when they directly support a recommendation.
6. Keep recommendations practical and manageable.
7. The caption must contain no more than 15 words.
8. The image prompt must represent hope and constructive progress.
9. Do not include graphic distress, self-harm, violence, or medical treatment.
10. Return valid JSON only.

Required JSON format:

{
  "empathetic_response": "string",
  "recommendations": [
    {
      "recommendation": "string",
      "source_title": "string or null",
      "page": "integer or null"
    }
  ],
  "caption": "string",
  "image_prompt": "string",
  "retrieval_status": "grounded, partially_grounded, or not_grounded",
  "sources": [
    {
      "title": "string",
      "page": "integer"
    }
  ]
}
""".strip()

Build the LLM prompt

In [25]:
def build_response_prompt(
    user_story,
    emotion_predictions,
    topic_result,
    retrieved_context,
    retrieved_sources,
):
    emotion_lines = []

    for prediction in emotion_predictions:
        label = prediction.get("emotion",prediction.get("label"))        

        score = prediction.get("score",prediction.get("probability"))      
            
        if score is None:
            emotion_lines.append(f"- {label}")
        else:
            emotion_lines.append(f"- {label}: "f"{float(score):.4f}")

    emotion_text = ("\n".join(emotion_lines) if emotion_lines else "No emotion prediction available.")

    primary_topic = topic_result.get("primary_topic","other")

    secondary_topic = (topic_result.get("secondary_topic") or "none")

    source_text = json.dumps(retrieved_sources,ensure_ascii=False,indent=2)

    return f""" USER STORY {user_story}
                EMOTIONS {emotion_text}
                PRIMARY TOPIC {primary_topic}
                SECONDARY TOPIC {secondary_topic}
                RETRIEVED KNOWLEDGE {retrieved_context}
                AVAILABLE SOURCES {source_text}
                Generate:
                    1. A 2-4 sentence empathetic acknowledgment.
                    2. Two or three practical recommendations.
                    3. A short encouraging caption.
                    4. A safe image-generation prompt.
                    5. Citations only when supported by the retrieved passages.
                Return valid JSON only.""".strip()

Generate the response

In [26]:
RESPONSE_MODEL = "gpt-4o-mini"

In [27]:
def generate_grounded_response( user_story,
                                emotion_predictions,
                                topic_result,
                                retrieved_context,
                                retrieved_sources,
                                model=RESPONSE_MODEL):
    
    prompt = build_response_prompt( user_story=user_story,
                                    emotion_predictions=(emotion_predictions),
                                    topic_result=topic_result,
                                    retrieved_context=(retrieved_context),
                                    retrieved_sources=(retrieved_sources))

    response = (client.chat.completions.create(model=model,
                                            temperature=0.3,
                                            response_format={"type": "json_object"},
                                            messages=[{"role": "system","content": (RESPONSE_SYSTEM_PROMPT)},  {"role": "user","content": prompt}]))
                       

    raw_content = (response.choices[0].message.content)

    return json.loads(raw_content)

**PART D — Combined RAG and response pipeline**

Create the combined function

In [28]:
def generate_emoexpress_output(user_story,
                                emotion_predictions,
                                topic_result,
                                vector_store,
                                retrieval_k=5):
    
    primary_topic = topic_result["primary_topic"]

    secondary_topic = (topic_result.get("secondary_topic"))

    emotion_labels = [prediction.get("emotion",prediction.get("label")) for prediction in (emotion_predictions)]

    retrieval_query = (build_retrieval_query(user_story=user_story,
                                            emotions=emotion_labels,
                                            primary_topic=primary_topic,
                                            secondary_topic=(secondary_topic)))

    retrieved_results, strategy = (retrieve_documents(vector_store=vector_store,
                                                    query=retrieval_query,
                                                    primary_topic=primary_topic,
                                                    k=retrieval_k))

    retrieved_context, sources = (format_retrieved_context(retrieved_results))

    if not retrieved_context.strip():
        retrieved_context = ("No directly relevant knowledge-base passages were retrieved.")
        sources = []

    generated_response = (generate_grounded_response(user_story=user_story,
                                                    emotion_predictions=(emotion_predictions),
                                                    topic_result=topic_result,
                                                    retrieved_context=(retrieved_context),
                                                    retrieved_sources=sources))

    generated_response["retrieval_strategy"] = strategy

    generated_response["retrieval_query"] = retrieval_query

    generated_response["retrieved_sources"] = sources

    return generated_response

**Part E — Test one example**

Sample inputs

In [29]:
sample_user_story = """I have applied to many jobs, but I keep getting rejected.
I am beginning to question whether I am capable enough.""".strip()

sample_emotion_predictions = [{"emotion": "disappointment","score": 0.82},{"emotion": "sadness","score": 0.69}]

sample_topic_result = {"primary_topic": ("career_and_work"),
                        "secondary_topic": ("self_confidence"),
                        "confidence": 0.94,}

Run the combined pipeline

In [30]:
complete_output = generate_emoexpress_output(user_story=sample_user_story,
                                            emotion_predictions=(sample_emotion_predictions),
                                            topic_result=sample_topic_result,
                                            vector_store=vector_store,
                                            retrieval_k=5)
complete_output

{'empathetic_response': "It's completely normal to feel disappointed after facing multiple job rejections. Remember, each rejection is a step toward finding the right opportunity, and it doesn't define your capabilities. You're not alone in this journey, and it's okay to seek support and reflect on your experiences.",
 'recommendations': [{'recommendation': 'Ask for feedback from interviewers to identify areas for improvement.',
   'source_title': 'Tips Job Search',
   'page': 3},
  {'recommendation': 'Celebrate small victories in your job search to maintain a positive mindset.',
   'source_title': 'Tips Job Search',
   'page': 4},
  {'recommendation': 'Reach out to friends or mentors for emotional support and encouragement.',
   'source_title': 'Stress Resilience',
   'page': 3}],
 'caption': 'Every rejection brings you closer to the right opportunity.',
 'image_prompt': 'A sunrise over a mountain, symbolizing new beginnings and hope.',
 'retrieval_status': 'grounded',
 'sources': [{'

Display the output

In [31]:
print("=" * 100)
print("EMPATHETIC RESPONSE")
print("=" * 100)
print(complete_output["empathetic_response"])

print("\nRECOMMENDATIONS")

for index, recommendation in enumerate(complete_output["recommendations"],start=1):
    print(f"{index}.{recommendation['recommendation']}"    )

    if recommendation.get("source_title"):
        print("   Source:",recommendation["source_title"], "| Page:",recommendation.get("page"))

print("\nCAPTION")
print(complete_output["caption"])

print("\nIMAGE PROMPT")
print(complete_output["image_prompt"])

print("\nRetrieval strategy:",complete_output["retrieval_strategy"])

EMPATHETIC RESPONSE
It's completely normal to feel disappointed after facing multiple job rejections. Remember, each rejection is a step toward finding the right opportunity, and it doesn't define your capabilities. You're not alone in this journey, and it's okay to seek support and reflect on your experiences.

RECOMMENDATIONS
1.Ask for feedback from interviewers to identify areas for improvement.
   Source: Tips Job Search | Page: 3
2.Celebrate small victories in your job search to maintain a positive mindset.
   Source: Tips Job Search | Page: 4
3.Reach out to friends or mentors for emotional support and encouragement.
   Source: Stress Resilience | Page: 3

CAPTION
Every rejection brings you closer to the right opportunity.

IMAGE PROMPT
A sunrise over a mountain, symbolizing new beginnings and hope.

Retrieval strategy: primary_topic


Save the generated output

In [33]:
output_path = (
    GENERATED_OUTPUT_DIR
    / "sample_rag_llm_output.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        complete_output,
        file,
        ensure_ascii=False,
        indent=4,
    )

print("Saved:", output_path)

Saved: d:\FullStack_Academic\Final Capstone\EmoExpress\outputs\generated_responses\sample_rag_llm_output.json


# Conclusion

This notebook combined the RAG knowledge-base workflow and LLM response generation.

The implementation loaded topic-organized PDF documents, preserved source and page metadata, split the documents into overlapping chunks, generated embeddings, and stored the chunks in a persistent Chroma vector database. The retrieval query combined the user story, detected emotions, primary topic, and secondary topic to search for relevant guidance.

The retrieved passages were then provided to the LLM to generate a structured EmoExpress response containing:

- An empathetic acknowledgment
- Practical recommendations
- An encouraging caption
- A safe image-generation prompt
- Source information with document titles and page numbers

This notebook also applied topic-based retrieval first, with fallback searches through general-support documents and the full knowledge base when necessary.

This approach improves the reliability of the recommendations because the LLM is grounded in retrieved PDF content rather than relying only on its general knowledge. The generated image prompt and caption can now be passed to the image-generation component in the next stage of the project.